# PointVisor – Point-Supervised Semantic Segmentation on DLRSD
**LandVisor Project Task Solution**

Implements partial Focal CE loss, simulates point annotations on real remote sensing data, trains a segmentation model, and runs experiments.

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import random
from glob import glob
from tqdm import tqdm
import pandas as pd
from skimage.exposure import match_histograms

DATA_ROOT = "DLRSD"
IMAGE_DIR = os.path.join(DATA_ROOT, "Images")
LABEL_DIR = os.path.join(DATA_ROOT, "Labels")

In [ ]:
# Verify paths
print("DATA_ROOT  :", DATA_ROOT)
print("IMAGE_DIR  :", IMAGE_DIR)
print("LABEL_DIR  :", LABEL_DIR)

# Define all possible image extensions
exts = ['*.jpg', '*.jpeg', '*.png', '*.tif', '*.tiff']
all_images = []

for ext in exts:
    # Search for both lowercase and uppercase versions
    all_images.extend(glob(os.path.join(IMAGE_DIR, "**", ext), recursive=True))
    all_images.extend(glob(os.path.join(IMAGE_DIR, "**", ext.upper()), recursive=True))

print(f"Total images found: {len(all_images)}")

if len(all_images) > 0:
    print("First 5 image paths:")
    for p in all_images[:5]:
        print("   ", p)
else:
    # If still 0, check what is actually in the directory
    print(f"\n[!] ALERT: No images found in {IMAGE_DIR}")
    print("Actual directory contents:", os.listdir(IMAGE_DIR)[:10])

In [ ]:
class PartialFocalLoss(nn.Module):
    """Partial Focal CE Loss"""
    def __init__(self, gamma=2.0, alpha=0.25, ignore_index=255):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.ignore_index = ignore_index

    def forward(self, pred, target, mask):
        # pred: (B, C, H, W) logits
        # target: (B, H, W) long
        # mask: (B, H, W) float 0/1 (only labeled points = 1)
        ce = nn.functional.cross_entropy(pred, target, reduction='none', ignore_index=self.ignore_index)
        pt = torch.exp(-ce)
        focal = self.alpha * (1 - pt) ** self.gamma * ce
        masked = focal * mask
        return masked.sum() / (mask.sum() + 1e-8)   # average only over labeled points

In [ ]:
class DLRSDPointDataset(Dataset):
    def __init__(self, image_files, points_per_class=5, reference_img_path=None):
        self.image_files = image_files
        self.points_per_class = points_per_class
        self.reference_img_path = reference_img_path if reference_img_path else image_files[0]

    def simulate_points(self, label_np):
        H, W = label_np.shape
        point_target = np.full((H, W), 255, dtype=np.int64) 
        point_mask = np.zeros((H, W), dtype=np.float32)

        # FIX: Find whatever unique values are ACTUALLY in the image (149, 160, etc)
        present_classes = [c for c in np.unique(label_np) if c != 255]
        
        for c in present_classes:
            ys, xs = np.where(label_np == c)
            n = min(self.points_per_class, len(ys))
            if n > 0:
                idx = random.sample(range(len(ys)), n)
                point_target[ys[idx], xs[idx]] = c
                point_mask[ys[idx], xs[idx]] = 1.0
        return point_target, point_mask

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        rel_path = os.path.relpath(img_path, IMAGE_DIR)
        base_name = os.path.splitext(rel_path)[0]
        label_path = os.path.join(LABEL_DIR, base_name + '.png')

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Style Stability
        ref_img = cv2.imread(self.reference_img_path)
        ref_img = cv2.cvtColor(ref_img, cv2.COLOR_BGR2RGB)
        img = match_histograms(img, ref_img, channel_axis=-1)

        img = img.astype(np.float32) / 255.0
        img = np.transpose(img, (2, 0, 1))

        label = cv2.imread(label_path, cv2.IMREAD_GRAYSCALE)
        
        # --- CRITICAL FIX: LABEL REMAPPING ---
        # Map [149, 160, ...] -> [0, 1, ...]
        unique_vals = sorted([c for c in np.unique(label) if c != 255])
        remapped_label = np.full(label.shape, 255, dtype=np.int64)
        for i, val in enumerate(unique_vals):
            if i < 17: # Stay within 17 classes
                remapped_label[label == val] = i
        label = remapped_label 
        # -------------------------------------

        point_target, point_mask = self.simulate_points(label)
        return torch.from_numpy(img), torch.from_numpy(label), torch.from_numpy(point_target), torch.from_numpy(point_mask)

    def __len__(self):
        return len(self.image_files)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = smp.Unet(encoder_name='resnet34', encoder_weights='imagenet', in_channels=3, classes=17).to(device)

criterion = PartialFocalLoss(gamma=2.0)
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

def train_one_epoch(dataloader, criterion, is_focal):
    model.train()
    total_loss = 0.0
    num_batches = 0

    for img, full_label, point_target, point_mask in tqdm(dataloader, desc="Training"):
        img = img.to(device)
        point_target = point_target.to(device)
        point_mask = point_mask.to(device)

        optimizer.zero_grad()
        pred = model(img)

        if is_focal:
            loss = criterion(pred, point_target, point_mask)
        else:
            loss = criterion(pred, point_target)  # vanilla CE ignores 255 automatically

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    return total_loss / num_batches if num_batches > 0 else 0.0

In [ ]:
# Use only first 300 images for fast experiments, it can be increasd
# Robust file collection for DLRSD subfolder structure
image_files = []
exts = ['*.jpg', '*.jpeg', '*.png', '*.tif', '*.tiff']
for ext in exts:
    image_files.extend(glob(os.path.join(IMAGE_DIR, "**", ext), recursive=True))
    image_files.extend(glob(os.path.join(IMAGE_DIR, "**", ext.upper()), recursive=True))

print(f"Total images found: {len(image_files)}")

# Use first 30 images for fast testing (increase to 300–1000+ later)
all_images = image_files[:30]

results = []

for n_points in [5, 15, 30]:
    for use_focal in [True, False]:
        model = smp.Unet(encoder_name='resnet34', encoder_weights='imagenet', in_channels=3, classes=17).to(device)
        optimizer = optim.AdamW(model.parameters(), lr=1e-3) # Higher LR to see movement

        dataset = DLRSDPointDataset(all_images, points_per_class=n_points)
        loader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=0)

        if use_focal:
            criterion = PartialFocalLoss(gamma=2.0)
        else:
            # reduction='sum' allows us to manually average only over labeled points
            criterion = nn.CrossEntropyLoss(ignore_index=255, reduction='sum')

        for epoch in range(5):
            model.train()
            total_loss = 0.0
            num_batches = 0

            for img, _, pt_target, pt_mask in tqdm(loader, desc=f"Pts:{n_points} Focal:{use_focal}"):
                img, pt_target, pt_mask = img.to(device), pt_target.to(device), pt_mask.to(device)

                optimizer.zero_grad()
                pred = model(img)

                if use_focal:
                    loss = criterion(pred, pt_target, pt_mask)
                else:
                    n_labeled = (pt_target != 255).sum()
                    if n_labeled == 0: continue
                    # Scale loss so it is visible
                    loss = criterion(pred, pt_target) / (n_labeled + 1e-8)

                loss.backward()
                optimizer.step()
                total_loss += loss.item()
                num_batches += 1

            avg_loss = total_loss / num_batches if num_batches > 0 else 0.0
            print(f"  Epoch {epoch+1} - Loss: {avg_loss:.6f}")

        results.append({"points_per_class": n_points, "focal": use_focal, "final_loss": avg_loss})

df = pd.DataFrame(results)
print("\nResults:")
print(df.round(4))
df.to_csv("experiment_results.csv", index=False)